# Init Lakehouse
Create schema, Volume, and upload source CSV files before running Bronze/Silver/Gold notebooks.

## Setup Connection

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]
VOLUME = os.environ.get("CLICKZETTA_VOLUME", "medallion_vol")

## Create Schema
In ClickZetta Lakehouse, a single schema holds all three medallion layers — no separate bronze/silver/gold schemas needed.

In [2]:
session.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}").collect()
print(f"Schema {SCHEMA!r} ready")

Schema 'public' ready


## Create Volume
The Volume stores raw CSV files for Bronze ingestion.

In [3]:
session.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.{VOLUME}").collect()
print(f"Volume {SCHEMA}.{VOLUME!r} ready")

Volume public.'medallion_vol' ready


## Upload Source CSV Files
Upload all CSV files from `datasets/engineering/` to the Volume.

In [4]:
from pathlib import Path
import os

# Locate project root regardless of where the notebook is launched from
# (project root contains the datasets/ directory)
_cwd = Path(os.getcwd())
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / "datasets").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASETS_DIR = PROJECT_ROOT / "datasets/engineering"
if not DATASETS_DIR.exists():
    raise FileNotFoundError(f"Cannot find datasets/engineering under {PROJECT_ROOT}")

for csv_path in sorted(DATASETS_DIR.rglob("*.csv")):
    rel    = csv_path.relative_to(DATASETS_DIR)
    subdir = str(rel.parent) if str(rel.parent) != "." else ""
    abs_path = str(csv_path.resolve())

    sql = (f"PUT '{abs_path}' TO VOLUME {VOLUME} SUBDIRECTORY '{subdir}'"
           if subdir else
           f"PUT '{abs_path}' TO VOLUME {VOLUME}")

    print(f"  PUT {rel} ...", end=" ", flush=True)
    session.sql(sql).collect()
    print("OK")

  PUT source_crm/cust_info.csv ... 

OK
  PUT source_crm/prd_info.csv ... 

OK
  PUT source_crm/sales_details.csv ... 

OK
  PUT source_erp/CUST_AZ12.csv ... 

OK
  PUT source_erp/LOC_A101.csv ... 

OK
  PUT source_erp/PX_CAT_G1V2.csv ... 

OK


## Verify

In [5]:
session.sql(f"LIST VOLUME {SCHEMA}.{VOLUME}").show()
print("Init complete — ready to run Bronze notebooks.")

+--------------------+---+-------+--------------------+
|       relative_path|url|   size|  last_modified_time|
+--------------------+---+-------+--------------------+
|source_crm/cust_i...|   | 855395|2026-05-26 16:26:...|
|source_crm/prd_in...|   |  26934|2026-05-26 16:26:...|
|source_crm/sales_...|   |3588116|2026-05-26 16:26:...|
|source_erp/CUST_A...|   | 561549|2026-05-26 16:26:...|
|source_erp/LOC_A1...|   | 402669|2026-05-26 16:26:...|
|source_erp/PX_CAT...|   |   1169|2026-05-26 16:26:...|
+--------------------+---+-------+--------------------+

Init complete — ready to run Bronze notebooks.
